# Web Scraping

## Billboard Scraping

The first step is to scrape the Billboard Global 200 chart for a given week.

In [372]:
import time
from operator import contains

#Import packages
import requests
import lxml.html as lx
import re
import pandas as pd
from datetime import datetime
import sqlite3 as sql
from requests.exceptions import HTTPError

In [2]:
#Set up base url and headers for scraping
base_url = 'https://www.billboard.com/charts'
headers = {
    'User-Agent': "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/26.0.1 Safari/605.1.15"
}

In [373]:
def get_html(link):
    try:
        billboard_info = requests.get(link, headers = headers)
        billboard_info.raise_for_status()
        billboard_html = lx.fromstring(billboard_info.text)
        return billboard_html
    except HTTPError as err:
        return None

In [3]:
def clean_string(stc):
    new_string = re.findall(r'(\S.*\S)', stc)
    if len(new_string) != 0:
        return new_string[0]
    else:
        return None

In [4]:
def clean_int(stc):
    new_int = re.sub(r'\s*', '', stc)
    return new_int

In [5]:
def clean_date(stc):
    str_date = re.findall(r"(?<=of )(.*)(?=\s)", stc)
    if len(str_date) != 0:
        str_date = str_date[0]
        new_date_format = datetime.strptime(str_date, "%B %d, %Y").date()
        reformatted_str_date = new_date_format.strftime("%m-%d-%Y")
        return reformatted_str_date
    else:
        return None

In [6]:
def get_billboard_data(billboard_html, row):

    billboard_row = {}
    billboard_row['rank'] = row
    billboard_row['song_name'] = clean_string(billboard_html.xpath(f"//ul[@data-detail-target='{row}']//h3/text()")[0])

    len_artist = len(billboard_html.xpath(f"//ul[@data-detail-target='{row}']//a"))
    if len_artist == 0:
        billboard_row['artist'] = clean_string(billboard_html.xpath(f"//ul[@data-detail-target='{row}']//li[@class='o-chart-results-list__item // lrv-u-flex-grow-1 lrv-u-flex lrv-u-flex-direction-column lrv-u-justify-content-center lrv-u-padding-l-050 lrv-u-padding-l-00@mobile-max u-max-width-397']//span/text()")[0])
        billboard_row['artist_link'] = None
    else:
        artists = billboard_html.xpath(f"//ul[@data-detail-target='{row}']//a/text()")[0]
        links = billboard_html.xpath(f"//ul[@data-detail-target='{row}']//a/@href")[0]
        for i in range(1, len_artist):
            artists = artists + ', ' + billboard_html.xpath(f"//ul[@data-detail-target='{row}']//a/text()")[i]
            links = links + ', ' + billboard_html.xpath(f"//ul[@data-detail-target='{row}']//a/@href")[i]
        billboard_row['artist'] = artists
        billboard_row['artist_link'] = links
    billboard_row['last_week_rank'] = clean_int(billboard_html.xpath(f"//ul[@data-detail-target='{row}']//div[@class='lrv-u-flex lrv-u-justify-content-space-between lrv-u-align-items-center u-grid-gap-125 u-align-self-stretch u-margin-b-0.375']//li//span")[0].text_content())
    billboard_row['peak'] = clean_int(billboard_html.xpath(f"//ul[@data-detail-target='{row}']//div[@class='lrv-u-flex lrv-u-justify-content-space-between lrv-u-align-items-center u-grid-gap-125 u-align-self-stretch u-margin-b-0.375']//li//span")[1].text_content())
    billboard_row['weeks_chart'] = clean_int(billboard_html.xpath(f"//ul[@data-detail-target='{row}']//div[@class='lrv-u-flex lrv-u-justify-content-space-between lrv-u-align-items-center u-grid-gap-125 u-align-self-stretch u-margin-b-0.375']//li//span")[2].text_content())
    billboard_row['week'] = clean_date(billboard_html.xpath("//div[@class='charts-title lrv-u-position-relative // lrv-u-width-100p u-padding-t-2.25 u-margin-b-0.625@tablet lrv-u-padding-t-050@mobile-max']//div[@class = 'u-padding-a-1@mobile-max u-padding-b-0.375@mobile-max']")[0].text_content())

    row_df = pd.DataFrame(billboard_row, index = [billboard_row["rank"]])
    return row_df

In [8]:
def get_billboard_chart(link):

    billboard_rows = []

    billboard_html = get_html(link)
    num_of_elements = len(billboard_html.xpath("//div[@class='o-chart-results-list-row-container']"))

    for i in range(1, num_of_elements + 1):
        new_row = get_billboard_data(billboard_html, i)
        billboard_rows.append(new_row)

    complete_chart = pd.concat(billboard_rows)

    return complete_chart

In [9]:
def get_billboard_chart_links(link):
    billboard_html = get_html(link)
    base_direc = 'https://www.billboard.com'

    charts_dic = {}
    chart_categories = billboard_html.xpath("//ul[@class='lrv-a-unstyle-list lrv-u-background-color-black lrv-u-color-white u-padding-b-3.125']//ul[@aria-labelledby='charts-menu-item-hits-of-the-world']//a/text()")
    chart_links = billboard_html.xpath("//ul[@class='lrv-a-unstyle-list lrv-u-background-color-black lrv-u-color-white u-padding-b-3.125']//ul[@aria-labelledby='charts-menu-item-hits-of-the-world']//a/@href")

    #US table
    hot_100_link = chart_links.append(billboard_html.xpath("//ul[@class='lrv-a-unstyle-list lrv-u-background-color-black lrv-u-color-white u-padding-b-3.125']//ul[@aria-labelledby='charts-menu-item-top-charts']//a/@href")[0])
    chart_categories.append('U.S. Hot 100 Songs')

    #Global 200
    hot_200_link = chart_links.append(billboard_html.xpath("//ul[@class='lrv-a-unstyle-list lrv-u-background-color-black lrv-u-color-white u-padding-b-3.125']//ul[@aria-labelledby='charts-menu-item-global']//a/@href")[0])
    chart_categories.append('Worldwide Top 200 Songs')

    for element, element_link in zip(chart_categories, chart_links):
        charts_dic[clean_string(element)] = base_direc + element_link

    return charts_dic

In [151]:
def get_all_billboard_global_charts(url):
    world_chart_links = get_billboard_chart_links(url)
    adjusted_world_charts = [chart_name for chart_name in list(world_chart_links.keys()) if re.search("Artist|Album|Global|Top Philippine|Thai Country|Vietnamese|Singles", chart_name) is None]

    all_charts = {}

    for chart in adjusted_world_charts:
        time.sleep(1)
        #print(chart)
        chart_link = world_chart_links[chart]
        chart_table = get_billboard_chart(chart_link)
        column_add = [chart] * chart_table.shape[0]
        chart_table.insert(loc = chart_table.shape[1], column = "chart", value = column_add)
        all_charts[chart] = chart_table

    return all_charts

In [152]:
all_charts_list_df = get_all_billboard_global_charts(base_url)

## Data Processing

In [182]:
all_charts_df = pd.concat(all_charts_list_df).reset_index(drop=True)
all_charts_df = all_charts_df.reindex(columns=["artist", "song_name", "rank", "artist_link", "last_week_rank", "peak", "weeks_chart", "week", "chart"])

In [183]:
all_charts_df

,artist,song_name,rank,artist_link,last_week_rank,peak,weeks_chart,week,chart
0,Flipperachi,FA9LA,1,None,1,1,10,02-28-2026,Billboard Arabic Hot 100
1,DYSTINCT,Yama,2,None,2,1,17,02-28-2026,Billboard Arabic Hot 100
2,Vanco Featuring AYA,Ma Tnsani,3,None,3,2,29,02-28-2026,Billboard Arabic Hot 100
3,"Lege-Cy, Ghaliaa",Msh Awl Marra,4,None,7,4,3,02-28-2026,Billboard Arabic Hot 100
4,DYSTINCT,Ta3al,5,None,4,3,4,02-28-2026,Billboard Arabic Hot 100
...,...,...,...,...,...,...,...,...,...
1645,Don Omar,Danza Kuduro,196,https://www.billboard.com/artist/don-omar/,-,140,37,02-28-2026,Worldwide Top 200 Songs
1646,Jimin,Who,197,https://www.billboard.com/artist/jimin/,-,1,74,02-28-2026,Worldwide Top 200 Songs
1647,Hozier,Too Sweet,198,https://www.billboard.com/artist/hozier/,-,1,98,02-28-2026,Worldwide Top 200 Songs
1648,"Shakira, Wyclef Jean",Hips Don't Lie,199,"https://www.billboard.com/artist/shakira/, htt...",-,134,20,02-28-2026,Worldwide Top 200 Songs


In [153]:
#Total number of mentions of artist in charts (does not account for collaboration)
def artist_nonunique_mentions(df):
    total_appearances_all_charts = df.groupby("artist").count().sort_values(by=['song_name'], ascending = False)[['song_name', 'week']]
    total_appearances_all_charts['week'] = df.loc[0, 'week']
    total_appearances_all_charts  = total_appearances_all_charts.rename(columns = {"song_name" : "count"}).reset_index()
    return total_appearances_all_charts

In [181]:
total_appearances_all_charts = artist_nonunique_mentions(all_charts_df)

In [156]:
def unique_artist_list(df):
    all_artists_combination = [str(entry).lower() for entry in total_appearances_all_charts['artist'].unique()]
    unique_artists = []
    for entry in all_artists_combination:
        entry_adjusted = re.sub(r'(\sx\s)|(feat.)|(\bft.\b)|(featuring)|(with)|(&)|(\+)', ', ', entry)
        all_artists_in_entry = re.split(r'[,]', entry_adjusted)
        for artist in all_artists_in_entry:
            no_whitespace = re.findall(r'(\w.*\w)', artist)
            if no_whitespace not in unique_artists and len(no_whitespace) >= 1:
                unique_artists.append(no_whitespace[0])
    return unique_artists

In [171]:
def artist_mentions_unique(df):
    unique_artists = unique_artist_list(df)

    individual_artist_chart_appearances_dict = {}
    for artist in unique_artists:
        artist_count = total_appearances_all_charts[total_appearances_all_charts['artist'].str.lower().str.contains(artist, regex = False)].sum(axis = 0).iloc[1]
        individual_artist_chart_appearances_dict[artist] = artist_count

    individual_artist_chart_appearances_df = pd.Series(individual_artist_chart_appearances_dict).to_frame().reset_index().rename(columns= {'index' : 'artist', 0 : 'count'})
    week = df.iloc[0, 2]
    individual_artist_chart_appearances_df['week'] = [week] * individual_artist_chart_appearances_df.shape[0]
    return individual_artist_chart_appearances_df

In [179]:
unique_artist_mentions = artist_mentions_unique(total_appearances_all_charts)

In [177]:
#How many charts each artist is charting in
def artist_chart_count(df):
    unique_chart_appearances = df.groupby(by = ["artist", "chart"]).count().groupby("artist").count().iloc[:, 0].sort_values(ascending=False).to_frame()
    unique_chart_appearances['week'] = df.loc[0, 'week']
    unique_chart_appearances = unique_chart_appearances.rename(columns = {"song_name" : "unique_count"}).reset_index()
    week = df.iloc[0, 7]
    unique_chart_appearances['week'] = [week] * unique_chart_appearances.shape[0]
    return unique_chart_appearances

In [180]:
unique_chart_appearances = artist_chart_count(all_charts_df)

## Streaming Scraping

In [353]:
spotify_url = 'https://kworb.net/spotify/'
apple_url = 'https://kworb.net'

In [354]:
def get_spotify_streaming_links(base_url_spotify):
    streaming_spotify_html = get_html('https://kworb.net/spotify/')
    charts_spotify_href = streaming_spotify_html.xpath("//table[@style='width: 410px;']//tr//@href")
    spotify_daily_charts_url = [(base_url_spotify + entry) for entry in charts_spotify_href if (not re.search('totals', entry)) and re.search('daily', entry)]
    spotify_weekly_charts_url = [(base_url_spotify + entry) for entry in charts_spotify_href if (not re.search('totals', entry)) and re.search('weekly', entry)]
    return spotify_daily_charts_url, spotify_weekly_charts_url

In [355]:
spotify_daily_links, spotify_weekly_links = get_spotify_streaming_links(spotify_url)

In [356]:
def get_apple_streaming_links(base_url_apple):
    streaming_apple_html = get_html('https://kworb.net/charts/')
    charts_apple_href = streaming_apple_html.xpath("//table[@style='width:1000px']//tr//@href")
    base_url_apple = 'https://kworb.net'
    apple_daily_charts_url = [(base_url_apple + entry) for entry in charts_apple_href if re.search('apple', entry)]
    return apple_daily_charts_url

In [357]:
apple_daily_links = get_apple_streaming_links(apple_url)

In [375]:
def get_streaming_charts_spotify(link_list):

    all_country_streaming_charts = []
    for link in link_list:
        print(link)
        #time.sleep(1)
        chart_html = get_html(link)
        if chart_html == None:
            continue
        table = pd.read_html(link, storage_options=headers)[0]
        table_name_content = chart_html.xpath("//div//span[@class='pagetitle']")[0].text_content()
        table_name_mid = re.findall(r'\S.*-', table_name_content)
        table_name = re.sub(r"- |-", '', table_name_mid[0])

        chart_date_info = chart_html.xpath("//div//span[@class='pagetitle']")[0].text_content()
        chart_date = re.findall(r'(\d.*\d)', chart_date_info)
        new_date_format = datetime.strptime(chart_date[0], "%Y/%m/%d").date()
        reformatted_str_date = new_date_format.strftime("%m-%d-%Y")

        table['song'] =
        table['artist'] =
        table['chart_name'] = [table_name] * table.shape[0]
        table['date'] = [reformatted_str_date] * table.shape[0]
        all_country_streaming_charts.append(table)

    all_country_streaming_charts_df = pd.concat(all_country_streaming_charts)

    return all_country_streaming_charts_df

In [376]:
def get_streaming_charts_apple(link_list):
    all_country_streaming_charts = []
    for link in link_list:
        #time.sleep(1)

        table = pd.read_html(link, storage_options=headers)[0]

        chart_html = get_html(link)
        if chart_html == None:
            continue
        table_name_content = chart_html.xpath("//div//span[@class='pagetitle']")[0].text_content()
        table_name = re.findall(r'(\S.*Songs)', table_name_content)[0]
        reformatted_str_date = datetime.now().strftime("%m-%d-%Y")

        table['chart_name'] = [table_name] * table.shape[0]
        table['date'] = [reformatted_str_date] * table.shape[0]
        all_country_streaming_charts.append(table)

    all_country_streaming_charts_df = pd.concat(all_country_streaming_charts)
    return all_country_streaming_charts_df

In [397]:
spotify_daily_streaming_charts = get_streaming_charts_spotify(spotify_daily_links)

https://kworb.net/spotify/country/global_daily.html
https://kworb.net/spotify/country/us_daily.html
https://kworb.net/spotify/country/gb_daily.html
https://kworb.net/spotify/country/ad_daily.html
https://kworb.net/spotify/country/ar_daily.html
https://kworb.net/spotify/country/au_daily.html
https://kworb.net/spotify/country/at_daily.html
https://kworb.net/spotify/country/by_daily.html
https://kworb.net/spotify/country/be_daily.html
https://kworb.net/spotify/country/bo_daily.html
https://kworb.net/spotify/country/br_daily.html
https://kworb.net/spotify/country/bg_daily.html
https://kworb.net/spotify/country/ca_daily.html
https://kworb.net/spotify/country/cl_daily.html
https://kworb.net/spotify/country/co_daily.html
https://kworb.net/spotify/country/cr_daily.html
https://kworb.net/spotify/country/cy_daily.html
https://kworb.net/spotify/country/cz_daily.html
https://kworb.net/spotify/country/dk_daily.html
https://kworb.net/spotify/country/do_daily.html
https://kworb.net/spotify/country/ec

In [ ]:
spotify_weekly_streaming_charts = get_streaming_charts_spotify(spotify_weekly_links)

In [ ]:
apple_daily_streaming_charts = get_streaming_charts_apple(apple_daily_links)

## Data Processing (Streaming)

In [404]:
def clean_spotify(all_country_streaming_charts_df):
    title = []
    artist = []
    for entry in all_country_streaming_charts_df["Artist and Title"]:
        new_entry = re.split(r' - ', entry)
        artist.append(new_entry[0])
        title.append(new_entry[1])
    all_country_streaming_charts_df['artist'] = artist
    all_country_streaming_charts_df['song_name'] = title

    all_country_streaming_charts_df = all_country_streaming_charts_df.rename(columns = {'Pos': "rank", "Pk": "peak"})

    all_country_streaming_charts_df = all_country_streaming_charts_df.reindex(columns=["rank", "artist", "song_name", "Days", "peak", "Streams", "Streams+", "7Day", "7Day+", "Total", "chart_name", "date"])
    return all_country_streaming_charts_df

In [ ]:
spotify_daily_streaming_charts = clean_spotify(spotify_daily_streaming_charts)

In [403]:
spotify_daily_streaming_charts

,rank,artist,song_name,days,peak,Streams,Streams+,7Day,7Day+,Total,chart_name,date
0,1,Bad Bunny,DtMF,NaN,1,6294134,-25798.0,46639878,-1118404,1542931869,Spotify Daily Chart Global,02-25-2026
1,2,Djo,End of Beginning,NaN,1,5017679,347105.0,32998062,190860,2219024575,Spotify Daily Chart Global,02-25-2026
2,3,PinkPantheress,Stateside + Zara Larsson (w/ Zara Larsson),NaN,3,4946464,763233.0,22849845,2500422,116877653,Spotify Daily Chart Global,02-25-2026
3,4,Bad Bunny,BAILE INoLVIDABLE,NaN,2,4532394,-17744.0,34029490,-924805,1269232978,Spotify Daily Chart Global,02-25-2026
4,5,Taylor Swift,The Fate of Ophelia,NaN,1,4524734,163010.0,30653319,-38296,1063377400,Spotify Daily Chart Global,02-25-2026
...,...,...,...,...,...,...,...,...,...,...,...,...
195,196,Orange,Khi Em Lớn (w/ Hoàng Dũng),NaN,3,20246,70.0,60997,20246,26733745,Spotify Daily Chart Vietnam,02-25-2026
196,197,Only C,Đau Để Trưởng Thành,NaN,98,20219,-3090.0,105298,20219,3021934,Spotify Daily Chart Vietnam,02-25-2026
197,198,Đen,Vị Nhà,NaN,75,20182,NaN,141737,-3026,3149006,Spotify Daily Chart Vietnam,02-25-2026
198,199,Ân ngờ,Vườn mây vừa hay (w/ MYLINA),NaN,86,20044,-3497.0,159510,1033,3438234,Spotify Daily Chart Vietnam,02-25-2026


## Save tables to Database

In [184]:
#Save as sql file
# Import/Create DB from CSV
sqlite_db_path = '../Final_Project/data.db' # Name of the table to be created in SQLite
conn = sql.connect(sqlite_db_path) # connect to a NEW database!

In [185]:
table_name_charts = 'BillboardCharts'
table_name_unique_artist_mentions = "UniqueArtistMentions"
table_name_all_artist_appearances = "AllArtistAppearances"
table_name_unique_artist_appearances = "UniqueArtistAppearances"

all_charts_df.to_sql(name=table_name_charts, con = conn, if_exists='replace', index=False)
unique_artist_mentions.to_sql(name=table_name_unique_artist_mentions, con = conn, if_exists='replace', index=False)
total_appearances_all_charts.to_sql(name=table_name_all_artist_appearances, con = conn, if_exists='replace', index=False)
unique_chart_appearances.to_sql(name=table_name_unique_artist_appearances, con = conn, if_exists='replace', index=False)

701

In [186]:
pd.read_sql('''SELECT * from sqlite_master''', conn)

,type,name,tbl_name,rootpage,sql
0,table,BillboardCharts,BillboardCharts,2,"CREATE TABLE ""BillboardCharts"" (\n""artist"" TEX..."
1,table,UniqueArtistMentions,UniqueArtistMentions,13,"CREATE TABLE ""UniqueArtistMentions"" (\n""artist..."
2,table,AllArtistAppearances,AllArtistAppearances,4,"CREATE TABLE ""AllArtistAppearances"" (\n""artist..."
3,table,UniqueArtistAppearances,UniqueArtistAppearances,9,"CREATE TABLE ""UniqueArtistAppearances"" (\n""art..."
